# Michigan Traders — Module 1
# NumPy for Quants  ·  *interactive workbook*

**Series:** MAT Education · Foundations
**Level:** Intermediate (you know Python; here we build the numerical muscle quant work runs on)
**Format:** **Interactive** — read the short intro, study the worked example, then **do the exercise yourself**.

---

Almost every quantitative idea you'll meet this year — returns, volatility, Sharpe ratios, covariance matrices, Monte-Carlo simulation, the math inside an ML model — is, under the hood, **array arithmetic**. NumPy is the library that makes that arithmetic fast, concise, and correct.

This is a **workbook**, not a lecture: every topic ends with a `# TODO` you fill in, a self-check cell that tells you instantly if you're right, and a hidden solution to peek at *only after* you've tried.

## How to use this notebook

Runs in a normal Jupyter kernel — no QuantConnect needed:

```bash
pip install numpy matplotlib
```

For each topic you'll see:

1. **A worked example** (already run — read the code and its output).
2. **✏️ Your turn** — a cell with `# TODO`. Replace the `None`/blank with your code.
3. **A self-check cell** — run it. If you see **`✅ Correct!`** you nailed it; an `AssertionError` tells you what's off.
4. **\U0001f4a1 Solution** — a collapsible block. Open it *after* attempting.

> **Rule of the club:** struggle with the TODO for a minute before revealing. That struggle is the learning.

Run the setup cell, then go top to bottom.

In [1]:
import numpy as np

rng = np.random.default_rng(42)   # seeded -> your numbers match the comments
np.set_printoptions(precision=4, suppress=True)
print("NumPy", np.__version__, "ready")

NumPy 2.5.0 ready


## 1. Why NumPy? Arrays vs. Python lists

A NumPy **array** holds one dtype in contiguous memory, so math runs as compiled, vectorized C — no Python loop. Two habits to build: **vectorize** (`a + b`, not a `for` loop) and remember arithmetic is **element-wise** by default.

In [2]:
import time
n = 1_000_000
py_list = list(range(n))
np_arr = np.arange(n)

t0 = time.perf_counter(); _ = [x * x for x in py_list]; t_list = time.perf_counter() - t0
t0 = time.perf_counter(); _ = np_arr * np_arr;          t_arr  = time.perf_counter() - t0
print(f"list loop : {t_list*1000:6.1f} ms")
print(f"numpy     : {t_arr*1000:6.1f} ms   ({t_list/t_arr:.0f}x faster, varies by machine)")

list loop :   25.2 ms
numpy     :    7.6 ms   (3x faster, varies by machine)


## 2. Creating arrays

`np.array`, `np.arange`, `np.linspace`, `np.zeros`, `np.ones`, `np.full`. A 2D array is your mental model for **rows = days, columns = assets**.

In [3]:
prices = np.array([100.0, 101.5, 99.8, 102.3, 103.1])
print("from list:", prices)
print("arange:   ", np.arange(0, 10, 2))
print("linspace: ", np.linspace(0, 1, 5))
print("full:     ", np.full(3, 0.05))          # e.g. a constant weight
print("random:   ", rng.normal(0, 1, size=4))

from list: [100.  101.5  99.8 102.3 103.1]
arange:    [0 2 4 6 8]
linspace:  [0.   0.25 0.5  0.75 1.  ]
full:      [0.05 0.05 0.05]
random:    [ 0.3047 -1.04    0.7505  0.9406]


### ✏️ Your turn — equal-weight portfolio

Build `weights`, an **equal-weight** vector over `n_assets = 5` (each weight = 1/5), using a NumPy call — not a Python list. It should sum to 1.

In [ ]:
n_assets = 5
weights = None   # TODO: an equal-weight NumPy vector of length n_assets

In [ ]:
assert weights is not None, "Replace None with your code."
assert weights.shape == (5,), f"Expected shape (5,), got {weights.shape}"
assert np.isclose(weights.sum(), 1.0), "Weights must sum to 1."
assert np.allclose(weights, 0.2), "Each weight should be 0.2."
print("✅ Correct!", weights)

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
n_assets = 5
weights = np.full(n_assets, 1 / n_assets)   # or: np.ones(n_assets) / n_assets
```
</details>

## 3. Shape, indexing & the view-vs-copy trap

`a.shape` is your debugging friend. Indexing extends to 2D as `a[rows, cols]`. **Gotcha:** a basic slice is a *view* — writing into it mutates the original. Use `.copy()` for independence.

In [6]:
data = np.arange(12).reshape(4, 3)   # 4 days, 3 assets
print("matrix:\n", data)
print("shape:", data.shape)
print("column (asset 2):", data[:, 2])   # all rows, col 2
print("row (day 1):     ", data[1, :])

p = np.array([100., 101., 102., 103.])
v = p[:2]; v[0] = -999
print("slice is a VIEW -> original changed:", p)

matrix:
 [[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]]
shape: (4, 3)
column (asset 2): [ 2  5  8 11]
row (day 1):      [3 4 5]
slice is a VIEW -> original changed: [-999.  101.  102.  103.]


### ✏️ Your turn — grab a price series

From `data = np.arange(12).reshape(4, 3)` (rows = days, cols = assets), extract **asset index 1's** full series across all days into `asset1`.

In [ ]:
data = np.arange(12).reshape(4, 3)
asset1 = None   # TODO: all days for the asset at column index 1

In [ ]:
assert asset1 is not None, "Replace None with your code."
assert np.array_equal(asset1, np.array([1, 4, 7, 10])), f"Got {asset1}"
print("✅ Correct! asset 1 series:", asset1)

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
asset1 = data[:, 1]   # ':' = every row (day), column index 1 (the asset)
```
</details>

## 4. Boolean masks — asking questions without loops

Build a boolean array (the *mask*), then select with it. This is how you find "all the down days," "all moves beyond 2σ," etc. `np.where(cond, a, b)` is a vectorized if/else.

In [9]:
r = np.array([0.012, -0.004, 0.0, 0.021, -0.018, 0.007])
mask = r < 0
print("down-day mask:", mask.astype(int))
print("num down days:", mask.sum())
print("the down returns:", r[mask])
print("signal (+1 up / -1 down):", np.where(r > 0, 1, -1))

down-day mask: [0 1 0 0 1 0]
num down days: 2
the down returns: [-0.004 -0.018]
signal (+1 up / -1 down): [ 1 -1 -1  1 -1  1]


### ✏️ Your turn — win rate

Given `rets` below, compute `win_rate` — the **fraction of strictly positive days** — as a single number. (Hint: a boolean array has a `.mean()`.)

In [ ]:
rets = np.array([0.01, -0.02, 0.005, -0.001, 0.03, -0.04, 0.012])
win_rate = None   # TODO: fraction of days with rets > 0

In [ ]:
assert win_rate is not None, "Replace None with your code."
assert np.isclose(win_rate, 4 / 7), f"Expected 4/7 = {4/7:.4f}, got {win_rate}"
print(f"✅ Correct! win rate = {win_rate:.1%}")

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
win_rate = (rets > 0).mean()   # True==1, False==0, so the mean is the fraction True
```
</details>

## 5. Vectorized math & broadcasting — prices → returns

The most fundamental transform in trading. **Broadcasting** stretches a smaller array to fit a bigger one (e.g. subtract a per-column mean from every row).

In [12]:
prices = np.array([100., 102., 101., 105., 107., 103.])
simple = prices[1:] / prices[:-1] - 1
print("simple returns:", simple)

# Broadcasting: z-score each column of a matrix
R = rng.normal(0, 0.01, size=(252, 4))
z = (R - R.mean(axis=0)) / R.std(axis=0)
print("z-scored col means ~0:", z.mean(axis=0))

simple returns: [ 0.02   -0.0098  0.0396  0.019  -0.0374]
z-scored col means ~0: [-0. -0.  0.  0.]


### ✏️ Your turn — log returns

Compute the **log returns** of `prices` into `log_ret` (length 5). Recall: log return $= \ln(P_t / P_{t-1})$. `np.diff` and `np.log` are useful.

In [ ]:
prices = np.array([100., 102., 101., 105., 107., 103.])
log_ret = None   # TODO: the log returns of `prices`

In [ ]:
assert log_ret is not None, "Replace None with your code."
assert log_ret.shape == (5,), f"Expected shape (5,), got {log_ret.shape}"
assert np.allclose(log_ret[0], 0.0198, atol=1e-4), "First log return should be ~0.0198."
assert np.allclose(log_ret, np.diff(np.log(prices))), "Values don't match log returns."
print("✅ Correct!", np.round(log_ret, 4))

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
log_ret = np.diff(np.log(prices))         # = np.log(prices[1:] / prices[:-1])
```
</details>

## 6. Aggregations and the `axis` argument

On a `(days, assets)` matrix: **`axis=0`** collapses rows → one number **per asset**; **`axis=1`** collapses columns → one number **per day**; no axis → a scalar. Getting `axis` right is half of practical NumPy.

In [15]:
R = rng.normal(0.0004, 0.01, size=(252, 4))
print("mean per asset (axis=0):", R.mean(axis=0))
print("mean per day   (axis=1) [:5]:", R.mean(axis=1)[:5])
print("annualized return per asset:", R.mean(axis=0) * 252)

mean per asset (axis=0): [-0.0004 -0.     -0.0012  0.0001]
mean per day   (axis=1) [:5]: [-0.0102  0.0006  0.0046 -0.0111  0.0065]
annualized return per asset: [-0.1131 -0.0102 -0.2909  0.0212]


### ✏️ Your turn — annualized volatility

Given the returns matrix `R` below (252 days × 3 assets), compute `ann_vol` = the **annualized volatility per asset** (daily std × √252). Shape should be `(3,)`.

In [ ]:
R = np.random.default_rng(1).normal(0, 0.01, size=(252, 3))
ann_vol = None   # TODO: annualized vol per asset

In [ ]:
assert ann_vol is not None, "Replace None with your code."
assert ann_vol.shape == (3,), f"Expected (3,), got {ann_vol.shape} - check your axis!"
assert np.allclose(ann_vol, R.std(axis=0) * np.sqrt(252)), "Values off - did you annualize with sqrt(252)?"
assert ((ann_vol > 0.10) & (ann_vol < 0.25)).all(), "Sanity: ~0.01 daily vol annualizes to ~0.16."
print("✅ Correct! annualized vol per asset:", np.round(ann_vol, 4))

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
ann_vol = R.std(axis=0) * np.sqrt(252)    # axis=0 -> per column (asset); sqrt(252) annualizes vol
```
</details>

## 7. Worked example: a strategy's stats (then your turn on drawdown)

The metrics every backtest report shows — annualized return/vol, **Sharpe**, **equity curve**, **drawdown** — are just a few NumPy lines. Study this, then implement max drawdown yourself.

In [18]:
rng2 = np.random.default_rng(9)
n_days, mu, sigma, dt = 252 * 3, 0.08, 0.20, 1 / 252
daily_log = rng2.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), n_days)
px = 100 * np.exp(np.cumsum(daily_log))
ret = px[1:] / px[:-1] - 1

ann_return = ret.mean() * 252
ann_vol = ret.std() * np.sqrt(252)
sharpe = ann_return / ann_vol
print(f"annualized return : {ann_return:6.2%}")
print(f"annualized vol    : {ann_vol:6.2%}")
print(f"Sharpe ratio      : {sharpe:6.2f}")

annualized return : 11.56%
annualized vol    : 19.95%
Sharpe ratio      :   0.58


### ✏️ Your turn — max drawdown

Given the `equity` curve below (growth of \$1), compute `max_dd`, the **worst peak-to-trough drawdown** (a negative number). Key tool: `np.maximum.accumulate(equity)` gives the running peak.

In [ ]:
equity = np.array([1.0, 1.1, 1.05, 1.2, 0.9, 1.0, 1.3])
max_dd = None   # TODO: most negative value of (equity / running_peak - 1)

In [ ]:
assert max_dd is not None, "Replace None with your code."
assert np.isclose(max_dd, -0.25), f"Expected -0.25 (0.9 down from a 1.2 peak), got {max_dd}"
print(f"✅ Correct! max drawdown = {max_dd:.1%}")

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
running_peak = np.maximum.accumulate(equity)   # highest equity seen so far
max_dd = (equity / running_peak - 1).min()     # drawdowns are <= 0; take the worst
```
</details>

## 8. Monte-Carlo simulation (then your turn on risk)

Generate **all** random draws at once into a matrix, then reduce — no path-by-path loop. Here: many 1-year price paths for one asset.

In [21]:
rng3 = np.random.default_rng(123)
n_paths, n_days = 10_000, 252
S0, mu, sigma, dt = 100.0, 0.07, 0.25, 1 / 252
shocks = rng3.normal((mu - 0.5 * sigma**2) * dt, sigma * np.sqrt(dt), size=(n_days, n_paths))
paths = S0 * np.exp(np.cumsum(shocks, axis=0))   # (252, 10000)
terminal = paths[-1]
terminal_return = terminal / S0 - 1
print(f"mean terminal price: {terminal.mean():.2f}")
print(f"5th percentile:      {np.percentile(terminal, 5):.2f}")

mean terminal price: 107.92
5th percentile:      68.77


### ✏️ Your turn — probability of loss

Using `terminal_return` from the simulation above, compute `prob_loss` — the **fraction of paths that ended below the starting price** (i.e. a negative return).

In [ ]:
prob_loss = None   # TODO: fraction of paths with terminal_return < 0

In [ ]:
assert prob_loss is not None, "Replace None with your code."
assert 0.0 <= prob_loss <= 1.0, "A probability must be in [0, 1]."
assert np.isclose(prob_loss, (terminal_return < 0).mean()), "Not quite - count negatives / total."
print(f"✅ Correct! P(loss over 1 year) = {prob_loss:.1%}")

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
prob_loss = (terminal_return < 0).mean()   # boolean mask -> fraction that are True
```
</details>

## 9. Portfolio linear algebra (then your turn on risk)

A portfolio's risk depends on how assets **co-move**: `portfolio variance = w · Σ · w`, where `Σ` is the covariance matrix. `@` is matrix multiply.

In [24]:
rng4 = np.random.default_rng(2024)
R = rng4.normal([0.0003, 0.0005, 0.0002], [0.008, 0.014, 0.006], size=(252, 3))
w = np.array([0.5, 0.3, 0.2])
cov = np.cov(R, rowvar=False)          # (3,3) daily covariance
print("correlation matrix:\n", np.round(np.corrcoef(R, rowvar=False), 2))
print("portfolio daily return series mean:", round((R @ w).mean(), 6))

correlation matrix:
 [[ 1.    0.02 -0.05]
 [ 0.02  1.    0.  ]
 [-0.05  0.    1.  ]]
portfolio daily return series mean: 0.000634


### ✏️ Your turn — portfolio volatility

Using `cov` and weights `w` above, compute `port_vol` — the **annualized** portfolio volatility: $\sqrt{w^\top \Sigma\, w}\times\sqrt{252}$.

In [ ]:
port_vol = None   # TODO: annualized portfolio volatility from cov and w

In [ ]:
assert port_vol is not None, "Replace None with your code."
ref = np.sqrt(w @ cov @ w) * np.sqrt(252)
assert np.isclose(port_vol, ref), f"Expected {ref:.4f}, got {port_vol}"
weighted_avg = w @ (R.std(axis=0) * np.sqrt(252))
assert port_vol < weighted_avg, "Diversified vol should be below the weighted-average vol."
print(f"✅ Correct! portfolio vol = {port_vol:.4f} (< weighted-avg {weighted_avg:.4f}: diversification!)")

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
port_vol = np.sqrt(w @ cov @ w) * np.sqrt(252)   # w @ cov @ w is the daily variance
```
</details>

## 10. The vectorization mindset — rank a whole universe at once

No loop over assets: compute every asset's Sharpe in one line, then rank with `argsort` (which returns **indices**).

In [27]:
universe = np.random.default_rng(11).normal(0.0003, 0.011, size=(252, 50))
sharpes = universe.mean(axis=0) / universe.std(axis=0) * np.sqrt(252)
print("computed", sharpes.size, "Sharpes, zero loops")
print("best asset index:", sharpes.argmax(), "  Sharpe:", round(sharpes.max(), 2))

computed 50 Sharpes, zero loops
best asset index: 21   Sharpe: 2.18


### ✏️ Your turn — top-5 by Sharpe

From `sharpes` above, build `top5` — the indices of the **5 highest-Sharpe assets, best first**. (Hint: `np.argsort` is ascending; slice the end and reverse.)

In [ ]:
top5 = None   # TODO: indices of the 5 highest-Sharpe assets, highest first

In [ ]:
assert top5 is not None, "Replace None with your code."
top5 = np.asarray(top5)
assert top5.shape == (5,), f"Expected 5 indices, got shape {top5.shape}"
assert top5[0] == sharpes.argmax(), "First element should be the best asset."
vals = sharpes[top5]
assert np.all(np.diff(vals) <= 0), "Should be sorted highest -> lowest."
print("✅ Correct! top-5 asset indices:", top5, " Sharpes:", np.round(vals, 2))

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
top5 = np.argsort(sharpes)[-5:][::-1]   # last 5 (highest) of ascending sort, then reverse
```
</details>

## \U0001f3c1 Mini-challenge: put it together

Write a reusable function. This is exactly the kind of helper you'll paste into research notebooks all year.

### ✏️ Your turn — a Sharpe function

Implement `annualized_sharpe(prices)` that takes a 1-D price array and returns the **annualized Sharpe ratio** (assume risk-free rate = 0). Steps: prices → simple returns → `mean/std × √252`.

In [ ]:
def annualized_sharpe(prices):
    # TODO: return the annualized Sharpe ratio (rf = 0)
    return None

In [ ]:
test_px = 100 * np.cumprod(1 + np.random.default_rng(5).normal(0.0005, 0.01, 500))
got = annualized_sharpe(test_px)
assert got is not None, "Return a value, not None."
_r = test_px[1:] / test_px[:-1] - 1
expected = _r.mean() / _r.std() * np.sqrt(252)
assert np.isclose(got, expected), f"Expected {expected:.4f}, got {got}"
print(f"✅ Correct! annualized Sharpe of the test series = {got:.2f}")

<details>
<summary>💡 <b>Solution</b> (try it yourself first!)</summary>

```python
def annualized_sharpe(prices):
    r = prices[1:] / prices[:-1] - 1
    return r.mean() / r.std() * np.sqrt(252)
```
</details>

## Cheat sheet

| Task | NumPy |
|---|---|
| Create | `np.array`, `np.arange`, `np.linspace`, `np.zeros`, `np.ones`, `np.full` |
| Random (seeded) | `rng = np.random.default_rng(seed)`; `rng.normal(...)` |
| Shape / slice | `a.shape`, `a[:, j]` (col), `a[i, :]` (row), `.copy()` |
| Mask / select | `a[a > 0]`, `(c1) & (c2)`, `np.where(cond, x, y)`, `mask.mean()` |
| Prices→returns | `p[1:]/p[:-1]-1` (simple); `np.diff(np.log(p))` (log) |
| Aggregate | `a.mean(axis=0)` per-asset, `axis=1` per-day |
| Cumulative | `np.cumsum`, `np.cumprod`, `np.maximum.accumulate` (running peak) |
| Portfolio | `R @ w`, `w @ cov @ w`, `np.cov(R, rowvar=False)` |
| Rank | `argmax`, `argsort` (return **indices**) |

**Golden rule:** writing a `for` loop over prices or assets? Ask whether a vectorized op does it instead. Usually it does.

## Stretch goals (bring questions to the next meeting)

1. Add a **risk-free rate** argument to `annualized_sharpe`.
2. Build a **2-asset efficient frontier**: sweep weight of asset A over `np.linspace(0, 1, 50)` and find the minimum-variance mix.
3. Re-run the Monte Carlo with `sigma = 0.40` — how does `prob_loss` change, and why?
4. Try a **20-day moving average** with pure NumPy slicing. Notice it's painful — that's why **Pandas** (Notebook 02) exists.

---

### What's next
**Notebook 02 — Pandas for Financial Data:** labeled, time-indexed data on top of these arrays (`DataFrame`, `DatetimeIndex`, `resample`, `rolling`, `pct_change`, `groupby`) — the bridge into QuantConnect's research environment.

**NumPy docs:** [Beginners guide](https://numpy.org/doc/stable/user/absolute_beginners.html) · [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · [Random generator](https://numpy.org/doc/stable/reference/random/generator.html)

*MAT Education · Foundations · Module 1.*